In [25]:
# excel 前兩行數據
# MallCode,	OR_Landmark_ID,	MallNameTc,	MallNameEn,	Id,	OR_POI_ID,	ReferenceId,	MallId,	Name_en,	Name_tc,	Name_sc,	CategoryNameEn,	ShopNumber
# tmtplaza,	42,	屯門市廣場,	tmtplaza,	1592,	803423,	540,	1,	&you,	&you,	&you,	Western Cuisine,	3212
# tmtplaza,	42,	屯門市廣場,	tmtplaza,	1605,	794645,	544,	1,	A-1 Bakery Premium,	A-1 Bakery Premium,	A-1 Bakery Premium,	Confectionery & Deli,	UG046

# 我的需求如下：

# 我將根據Sino_poi_1.sql的代碼生成幾個列表（poi list）。現在我要讀取 excel 中的 每個landmark的所有POI 數據。
# 我要(繼續為landmark生成相應的列表）將每個landmark下的poi進行分類（Booking/Voucher/Takeaway/Retail Video Views），邏輯請複刻 Sino_poi_1.sql，請你幫我生成相應的代碼。
# 請保持SET @landmarkId = 37 的設定，我可以自己輸入landmark id生成結果。
# 有些orpoiid是空白的，這種情況請你refer to excel的Id欄，這是mars.dbo.poi的poiid （另外，mars.dbo.poi的orpoiid= openrice3.dbo.poi的poiid
# 有幾個landmarkid是NA，但是有商戶的orpoiid信息，請你給我一個單獨的input地方（類似landmark_id = 37）讓我可以在處理這些地點時手動輸入poi id進行後續分類，cell2是excel例子。

In [26]:
# MallCode,	OR_Landmark_ID,	MallNameTc,	MallNameEn,	Id,	OR_POI_ID,	ReferenceId,	MallId,	Name_en,	Name_tc,	Name_sc,	CategoryNameEn,	ShopNumber
# wanchai	N/A	灣仔	Wan Chai	1938	756721		12	Sugidama (The Hennessy)	杉玉居酒屋 (The Hennessy)	杉玉居酒屋 (The Hennessy)	Asian Cuisine	B&C
# wanchai	N/A	灣仔	Wan Chai	1943	31166		12	Wooloomooloo Steakhouse (The Hennessy)	Wooloomooloo Steakhouse (The Hennessy)	Wooloomooloo Steakhouse (The Hennessy)	Western Cuisine	1


In [27]:
import pandas as pd
import pymssql

DB_info = {'server': '192.168.61.119:7622', 'user': 'BAReporting', 'password': 'KeHeCReme8he'}
landmark_id = 9378

manual_or_poi_ids = []        # LandmarkId = N/A, input orpoiid: [756721, 31166]
start_date = '2026-09-01'
end_date = '2026-10-01'

poi_source = pd.read_excel('Sino POI full list.xlsx', sheet_name='Sino POIs')
poi_data = (
    poi_source.rename(columns={
        'OR Landmark ID': 'LandmarkId',
        'OR POI ID': 'ORPoiId',
        'Id': 'MarsPoiId',
    })
    .assign(
        LandmarkId=lambda data: pd.to_numeric(data['LandmarkId'], errors='coerce'),
        ORPoiId=lambda data: pd.to_numeric(data['ORPoiId'], errors='coerce'),
        MarsPoiId=lambda data: pd.to_numeric(data['MarsPoiId'], errors='coerce'),
    )
)

landmark_pois = (
    poi_data.dropna(subset=['LandmarkId'])
    .dropna(subset=['ORPoiId', 'MarsPoiId'], how='all')
    .astype({'LandmarkId': 'int64', 'ORPoiId': 'Int64', 'MarsPoiId': 'Int64'})
    .loc[lambda data: data['LandmarkId'].eq(landmark_id), ['LandmarkId', 'ORPoiId', 'MarsPoiId']]
)
manual_pois = pd.DataFrame({
    'LandmarkId': landmark_id,
    'ORPoiId': manual_or_poi_ids,
    'MarsPoiId': pd.NA,
}).astype({'LandmarkId': 'int64', 'ORPoiId': 'Int64', 'MarsPoiId': 'Int64'})

selected_pois = (manual_pois if manual_or_poi_ids else landmark_pois).drop_duplicates()

if selected_pois.empty:
    raise ValueError(f'No POIs selected for landmark_id={landmark_id}.')

selected_pois

,LandmarkId,ORPoiId,MarsPoiId
256,9378,778393,1838
257,9378,727670,1708
258,9378,668541,1419
259,9378,668175,1420
260,9378,746108,1414
261,9378,642090,1427
262,9378,463876,1421
263,9378,12821,1423
264,9378,746817,1422
265,9378,1074,1424


In [28]:
excel_poi_values = ', '.join(
    f"({int(or_poi_id) if pd.notna(or_poi_id) else 'NULL'}, "
    f"{int(mars_poi_id) if pd.notna(mars_poi_id) else 'NULL'})"
    for or_poi_id, mars_poi_id in selected_pois[['ORPoiId', 'MarsPoiId']].itertuples(index=False)
)

sql = f'''
SET NOCOUNT ON;

DECLARE @START datetime = %s;
DECLARE @END datetime = %s;

CREATE TABLE #ExcelPoi (
    ORPoiId int NULL,
    MarsPoiId int NULL
);

INSERT INTO #ExcelPoi (ORPoiId, MarsPoiId)
VALUES {excel_poi_values};

SELECT DISTINCT
    mars_poi.PoiId AS MarsPoiId,
    mars_poi.ORPoiId,
    mars_poi.NameLang1 AS PoiName
INTO #SelectedPoi
FROM mars.dbo.Poi AS mars_poi
INNER JOIN #ExcelPoi AS excel_poi
    ON (excel_poi.ORPoiId = mars_poi.ORPoiId AND mars_poi.RegionId = 0)
    OR (excel_poi.ORPoiId IS NULL AND excel_poi.MarsPoiId = mars_poi.PoiId);

SELECT
    selected.MarsPoiId,
    VideoViews = COALESCE(SUM(CASE WHEN poi.PoiTypeId = 10 THEN media.HitCount ELSE 0 END), 0)
INTO #PoiVideoViews
FROM #SelectedPoi AS selected
LEFT JOIN openrice3.dbo.Media AS media ON media.PoiId = selected.ORPoiId
LEFT JOIN openrice3.dbo.Poi AS poi ON poi.PoiId = media.PoiId
GROUP BY selected.MarsPoiId;

CREATE TABLE #CategorizedPoi (
    Category varchar(10) NOT NULL,
    MarsPoiId int NOT NULL,
    ORPoiId int NULL,
    PoiName nvarchar(255) NULL
);

INSERT INTO #CategorizedPoi (Category, MarsPoiId, ORPoiId, PoiName)
SELECT DISTINCT
    'Booking', selected.MarsPoiId, selected.ORPoiId, selected.PoiName
FROM #SelectedPoi AS selected
INNER JOIN mars.dbo.Booking AS booking ON booking.PoiId = selected.MarsPoiId
INNER JOIN mars.dbo.BizService AS service ON service.PoiId = selected.MarsPoiId
WHERE booking.BookingTime >= @START
  AND booking.BookingTime < @END
  AND booking.[Status] = 10;

INSERT INTO #CategorizedPoi (Category, MarsPoiId, ORPoiId, PoiName)
SELECT DISTINCT
    'Voucher', selected.MarsPoiId, selected.ORPoiId, selected.PoiName
FROM #SelectedPoi AS selected
INNER JOIN mars.dbo.BizService AS service ON service.PoiId = selected.MarsPoiId
WHERE service.ServiceStartTime < @END
  AND service.ServiceEndTime >= @END
  AND service.ServiceTypeId = 5;

DECLARE @MIN_TakeAwayOrderID int;

SELECT @MIN_TakeAwayOrderID = MAX(TakeAwayOrderID)
FROM mars.dbo.TakeAwayOrder WITH (NOLOCK)
WHERE CreateTime < @START;

INSERT INTO #CategorizedPoi (Category, MarsPoiId, ORPoiId, PoiName)
SELECT DISTINCT
    'Takeaway', selected.MarsPoiId, selected.ORPoiId, selected.PoiName
FROM #SelectedPoi AS selected
INNER JOIN mars.dbo.TakeAwayOrder AS takeaway WITH (NOLOCK)
    ON takeaway.PoiId = selected.MarsPoiId
INNER JOIN mars.dbo.BizService AS service ON service.PoiId = selected.MarsPoiId
WHERE takeaway.TakeAwayOrderId >= @MIN_TakeAwayOrderID
  AND takeaway.CreateTime < @END
  AND takeaway.[Status] = 10
  AND service.ServiceStartTime < @END
  AND service.ServiceEndTime >= @END
  AND service.ServiceTypeId = 4;

SELECT
    categorized.Category,
    categorized.MarsPoiId,
    categorized.ORPoiId,
    categorized.PoiName,
    video.VideoViews
FROM #CategorizedPoi AS categorized
INNER JOIN #PoiVideoViews AS video ON video.MarsPoiId = categorized.MarsPoiId

UNION ALL

SELECT
    Category = CAST(NULL AS varchar(10)),
    selected.MarsPoiId,
    selected.ORPoiId,
    selected.PoiName,
    video.VideoViews
FROM #SelectedPoi AS selected
INNER JOIN #PoiVideoViews AS video ON video.MarsPoiId = selected.MarsPoiId
LEFT JOIN #CategorizedPoi AS categorized ON categorized.MarsPoiId = selected.MarsPoiId
WHERE categorized.MarsPoiId IS NULL

UNION ALL

SELECT
    Category = CAST(NULL AS varchar(10)),
    excel_poi.MarsPoiId,
    excel_poi.ORPoiId,
    COALESCE(openrice_poi.NameLang1, N'(not found in Mars)') COLLATE DATABASE_DEFAULT,
    VideoViews = COALESCE(SUM(CASE WHEN openrice_poi.PoiTypeId = 10 THEN media.HitCount ELSE 0 END), 0)
FROM #ExcelPoi AS excel_poi
LEFT JOIN #SelectedPoi AS selected
    ON (excel_poi.ORPoiId = selected.ORPoiId)
    OR (excel_poi.ORPoiId IS NULL AND excel_poi.MarsPoiId = selected.MarsPoiId)
LEFT JOIN openrice3.dbo.Poi AS openrice_poi ON openrice_poi.PoiId = excel_poi.ORPoiId
LEFT JOIN openrice3.dbo.Media AS media ON media.PoiId = excel_poi.ORPoiId
WHERE selected.MarsPoiId IS NULL
GROUP BY excel_poi.MarsPoiId, excel_poi.ORPoiId, openrice_poi.NameLang1
ORDER BY Category, PoiName;
'''

with pymssql.connect(**DB_info) as conn:
    all_poi_results = pd.read_sql(sql, conn, params=(start_date, end_date))

poi_lists = all_poi_results.loc[all_poi_results['Category'].notna()].reset_index(drop=True)
unclassified_pois = all_poi_results.loc[all_poi_results['Category'].isna()].reset_index(drop=True)

booking_pois = poi_lists.loc[poi_lists['Category'].eq('Booking')].reset_index(drop=True)
voucher_pois = poi_lists.loc[poi_lists['Category'].eq('Voucher')].reset_index(drop=True)
takeaway_pois = poi_lists.loc[poi_lists['Category'].eq('Takeaway')].reset_index(drop=True)

if unclassified_pois.empty:
    print('Category = NA: no unclassified POIs.')
else:
    print(f'Category = NA ({len(unclassified_pois)} POIs)')
    print(unclassified_pois.to_string(index=False))

poi_lists

C:\Users\lenalee\AppData\Local\Temp\ipykernel_38032\2485486871.py:128: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  all_poi_results = pd.read_sql(sql, conn, params=(start_date, end_date))


Category = NA (17 POIs)
Category  MarsPoiId  ORPoiId       PoiName  VideoViews
     NaN      34993   778393 Bistro Hoi An           0
     NaN       1708   727670   Chef's Cuts           0
     NaN      32016   746108   Green Basil           0
     NaN      30860   746817       Le Cafe           0
     NaN       2320  1020917         Nonna           0
     NaN       2238     3589          大堂酒吧           0
     NaN      32015    31999         意大利餐廳           0
     NaN      28379   668175           星小宴           0
     NaN       1416   657016 星庭 新加坡海南雞飯專門店           0
     NaN       1419   668541       正好燒鵝茶餐廳           0
     NaN        499    37781           沙嗲軒           0
     NaN         78    12821          法國飯店           0
     NaN       1427   642090            海圖           0
     NaN       7198   491690          海岸扒房           0
     NaN        500    26117             粵           0
     NaN       2145   463876       金功夫點心小廚           0
     NaN       7482     1074           麥當

,Category,MarsPoiId,ORPoiId,PoiName,VideoViews
0,Booking,2237,26117,粵,0
1,Booking,2236,235,聆渢咖啡廳,0
2,Takeaway,26278,692629,紅葉茶屋,0
